# 02 · REAL data for Indian waters — Global Fishing Watch — 5 YEARS (2021–2025)

This notebook downloads **real** data for the Gulf of Mannar – Palk Bay pilot (`IN-GOM`) from the Global Fishing Watch (GFW) API and puts it into the project pipeline. No synthetic data is used anywhere here.

| What | Source (real) |
|---|---|
| Apparent fishing events, encounters (vessel meetings), loitering, port visits, **AIS-off gaps** (with off/on positions and satellite-reception quality) | GFW Events API, derived from real AIS |
| Vessel detections from **Sentinel-1 radar** that match **no AIS vessel** (candidate "dark" vessels) | GFW 4Wings SAR presence |
| Apparent fishing hours per vessel | GFW 4Wings fishing effort |

**Important limits**
- **No raw ping-by-ping AIS tracks:** GFW's public API gives processed events instead.
- **Incomplete coverage:** many small Indian fishing boats carry no AIS, so real coverage is partial.
- **No labels:** there are **no public "confirmed illegal fishing" labels** for these waters. Everything here describes *activity patterns and investigation evidence*, never illegality.

**Terms:** non-commercial use only; attribute *Global Fishing Watch*. Vessel identities are anonymised in every output.

**Runtime:** use a CPU runtime. The download for the default wide area (southern Indian EEZ, 2021–2025, 5 years) mostly waits on the API. Months already cached from an earlier shorter run (for example 2024–2025) are reused, so only the new months download. Months download 4 at a time and each finished month is cached, so if the session stops, **Run all** resumes where it left off. Downloads are cached month by month, so a re-run resumes where it stopped.

In [ ]:
# ---- Setup: works in Google Colab (CPU is enough) and locally --------------------
import os, sys, subprocess, pathlib, zipfile, glob
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Dan-droid-A-113/IUU-Fishing-GNN-India"   # clones the code; set to "" to upload the zip instead
USE_DRIVE = False      # True = keep project + outputs on Google Drive (restart-safe)

if IN_COLAB:
    PROJECT_DIR = pathlib.Path("/content/india-fishing-gnn")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        PROJECT_DIR = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn")
    if not (PROJECT_DIR / "src").exists():
        if REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, str(PROJECT_DIR)], check=True)
        else:
            # accepts renamed downloads such as "india-fishing-gnn (1).zip"
            zips = sorted(glob.glob("/content/india-fishing-gnn*.zip"), key=os.path.getmtime)
            if not zips:
                from google.colab import files
                print("Project zip not found - choose india-fishing-gnn.zip from your computer:")
                uploaded = files.upload()   # file picker appears below this cell
                zips = [os.path.abspath(n) for n in uploaded if n.endswith(".zip")]
            if not zips:
                raise FileNotFoundError("No project zip uploaded.")
            print("Extracting", zips[-1])
            zipfile.ZipFile(zips[-1]).extractall(PROJECT_DIR.parent)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-colab.txt")], check=True)
else:
    PROJECT_DIR = pathlib.Path.cwd()
    while not (PROJECT_DIR / "src").exists() and PROJECT_DIR != PROJECT_DIR.parent:
        PROJECT_DIR = PROJECT_DIR.parent

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR, "| Colab:", IN_COLAB)

## 0 · Your API token (kept private)

1. In Colab, click the **key icon 🔑** in the left sidebar (*Secrets*).
2. Click **Add new secret**. Name: `GFW_API_TOKEN`. Value: paste your token.
3. Switch **Notebook access** on.

The token is never printed or saved to any file. If no secret is found, the cell asks for it in a hidden input box.

In [ ]:
from src.gfw_real import get_token, make_client
client = make_client(get_token())
print("GFW client ready (token loaded, not displayed).")

## 1 · Settings

In [ ]:
import json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore")
from src.config import REGIONS, PROCESSED_DIR, FIGURE_DIR, METRIC_DIR
from src.gfw_real import (GFW_DIR, EVENT_DATASETS, REAL_CAVEAT, bbox_polygon, fetch_events, fetch_sar,
                          fetch_fishing_effort, fetch_region_names, anonymize_real, sar_near_gaps,
                          vessel_week_tables, write_provenance)

# Wider real-data area: southern Indian EEZ (Kerala, Lakshadweep Sea, Tamil Nadu, Gulf of Mannar - Palk Bay).
# For the small pilot box only, use: REGION_ID = "IN-GOM"; BBOX = REGIONS["IN-GOM"].bbox
REGION_ID = "IN-SOUTH"
REGION_NAME = "Southern Indian EEZ (Kerala, Lakshadweep Sea, Tamil Nadu, Gulf of Mannar)"
BBOX = (5.0, 16.0, 70.0, 82.0)       # (lat_min, lat_max, lon_min, lon_max)
START, END = "2021-01-01", "2025-12-31"   # 5 years (was 2024-2025); GFW data run up to ~3 days ago
# Order matters: rare, important events first; bulky fishing / port-visit events last.
KINDS = ["gap", "encounter", "loitering", "fishing", "port_visit"]
GEOM = bbox_polygon(BBOX)
REAL_DIR = PROCESSED_DIR / "real_gfw"
# In Colab, keep the month-by-month download cache on Google Drive, so months already
# downloaded in an earlier run (e.g. 2024-2025) are reused and only new months download.
DRIVE_DATA = pathlib.Path("/content/drive/MyDrive/india-fishing-gnn-data")
CACHE = GFW_DIR
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = DRIVE_DATA / "gfw_raw_cache"
for p in (CACHE, REAL_DIR, FIGURE_DIR, METRIC_DIR):
    p.mkdir(parents=True, exist_ok=True)
_cached = sorted(CACHE.glob("events_fishing_*.parquet"))
print(f"download cache: {CACHE} ({len(_cached)} fishing-event months already cached)")
print(REAL_CAVEAT)
print("region:", REGION_NAME, BBOX, "| period:", START, "->", END)

## 2 · Download real events (cached per month)

In [ ]:
raw_events, failures = {}, {}
for kind in KINDS:
    print(f"Fetching {kind} events ...")
    try:
        raw_events[kind] = await fetch_events(client, kind, GEOM, START, END, cache_dir=CACHE)
    except Exception as ex:
        failures[kind] = f"{type(ex).__name__}: {ex}"
        print("   FAILED:", failures[kind])
display(pd.Series({k: len(v) for k, v in raw_events.items()}, name="events").to_frame())
if failures:
    print("Some requests failed (see above). Re-run this cell later; finished months are cached.")

## 3 · Download real Sentinel-1 radar detections (unmatched to AIS) and fishing effort

In [ ]:
try:
    sar = await fetch_sar(client, GEOM, START, END, unmatched_only=True, cache_dir=CACHE)
except Exception as ex:
    print("SAR download failed:", ex); sar = pd.DataFrame(columns=["date", "lat", "lon", "detections"])
try:
    effort = await fetch_fishing_effort(client, GEOM, START, END, cache_dir=CACHE)
except Exception as ex:
    print("Fishing-effort download failed:", ex); effort = pd.DataFrame()
print(f"SAR cell-days without AIS match: {len(sar):,} (total detections {int(sar.detections.sum()) if len(sar) else 0:,})")
print(f"fishing-effort rows: {len(effort):,}")

## 4 · Combine, interpret regions, anonymise, save

In [ ]:
names = await fetch_region_names(client)
ev_raw = pd.concat([v for v in raw_events.values() if len(v)], ignore_index=True)
# keep events that overlap the study period (long port visits can start years earlier)
ev_raw = ev_raw[ev_raw.end.fillna(ev_raw.start) >= pd.Timestamp(START, tz="UTC")].reset_index(drop=True)
ev, id_map = anonymize_real(ev_raw)
id_map.to_parquet(REAL_DIR / "_private_id_map.parquet")          # private, git-ignored
eez_names = names.get("eez", {})
india_eez = [k for k, v in eez_names.items() if "india" in v.lower() and "indian ocean" not in v.lower()]
print("EEZ ids treated as Indian:", {k: eez_names[k] for k in india_eez} or "none found (lookup unavailable)")
ev["eez_names"] = ev.eez_ids.map(lambda s: ", ".join(eez_names.get(x, x) for x in s.split(",") if x))
ev["mpa_names"] = ev.mpa_ids.map(lambda s: ", ".join(names.get("mpa", {}).get(x, x) for x in s.split(",") if x))
gi = ev.event_type == "gap"
ev.loc[gi, "sar_dark_nearby"] = sar_near_gaps(ev[gi], sar, radius_km=20).values
ev.to_parquet(REAL_DIR / "events.parquet", index=False)
sar.to_parquet(REAL_DIR / "sar_unmatched.parquet", index=False)
if len(effort):
    effort.drop(columns=[c for c in ("ship_name", "mmsi", "imo", "call_sign") if c in effort]).to_parquet(
        REAL_DIR / "fishing_effort.parquet", index=False)
prov = write_provenance(REAL_DIR / "provenance.json", REGION_NAME, BBOX, START, END,
                        {**{k: len(v) for k, v in raw_events.items()}, "sar_cell_days": len(sar), "effort_rows": len(effort)})
print(json.dumps(prov, indent=2)[:1200])

## 5 · What is in the real data?

In [ ]:
print(f"vessels: {ev.vessel_id.nunique():,} | events: {len(ev):,} | {ev.start.min():%Y-%m-%d} -> {ev.start.max():%Y-%m-%d}")
display(pd.crosstab(ev.start.dt.tz_convert(None).dt.to_period("M"), ev.event_type))
display(ev.groupby("event_type").vessel_type.value_counts().unstack(fill_value=0))
display(ev.flag.value_counts().head(10).rename("events by flag").to_frame())
g = ev[ev.event_type == "gap"]
if len(g):
    display(pd.Series({"AIS-off gaps": len(g),
                       "flagged intentional by GFW": int(g.intentional_disabling.fillna(False).sum()),
                       "median gap (h)": g.gap_duration_h.median(),
                       "gaps with unmatched SAR detections within 20 km": int((g.sar_dark_nearby > 0).sum()),
                       "gaps starting or ending inside an MPA": int(g.in_mpa.sum())}, name="value").to_frame())
e = ev[ev.event_type == "encounter"]
if len(e):
    display(e.encounter_type.value_counts().rename("encounters").to_frame())
display(ev[ev.in_mpa].groupby(["mpa_names", "event_type"]).size().rename("events inside MPAs").to_frame().head(20))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
m = pd.crosstab(ev.start.dt.tz_convert(None).dt.to_period("M"), ev.event_type)
m.plot(ax=ax, marker="o", lw=1.6)
ax.set_ylabel("events per month"); ax.set_xlabel(""); ax.set_title("Real GFW events per month - pilot region", loc="left", fontsize=10)
ax.grid(color="#e9e8e4", lw=0.5); plt.tight_layout(); plt.savefig(FIGURE_DIR / "real_events_per_month.png", dpi=150); plt.show()

## 6 · Maps of the real data

In [ ]:
from src.india_layers import get_layers
from src import visualization as viz
layers = get_layers("IN-GOM")
# static map zooms to the Gulf of Mannar pilot; the interactive map below shows the whole downloaded area
fig = viz.plot_real_events_map(ev, sar, layers, f"Real GFW events and SAR detections - Gulf of Mannar / Palk Bay zoom ({START} to {END})",
                               FIGURE_DIR / "real_events_map.png")
plt.show()
m = viz.folium_real_map(ev, sar, layers, FIGURE_DIR / "real_events_map.html")
m

## 7 · Real vessel-week interaction graph (input for the graph model)

- **Nodes:** vessel × week, with features aggregated from real events:
  - fishing hours, split into inside MPAs and outside Indian waters;
  - AIS-off gaps: counts, hours, GFW's intentional-disabling flag and satellite reception;
  - unmatched SAR detections near gaps;
  - loitering, encounters (with carriers counted separately) and port visits.
- **Edges:** real encounters between vessels, plus each vessel's previous active week.

In [ ]:
nodes, edges = vessel_week_tables(ev, india_eez_ids=india_eez)
nodes.to_parquet(REAL_DIR / "graph_nodes_vessel_week.parquet", index=False)
edges.to_parquet(REAL_DIR / "graph_edges.parquet", index=False)
display(pd.Series({"nodes (vessel-weeks)": len(nodes), "vessels": nodes.vessel_id.nunique(),
                   "encounter edges (directed)": int((edges.edge_type == "encounter").sum()),
                   "temporal edges": int((edges.edge_type == "temporal").sum())}, name="count").to_frame())
cols = ["fishing_hours", "fishing_in_mpa_hours", "fishing_outside_india_eez_hours", "n_gaps", "gap_hours",
        "n_intentional_gaps", "sar_dark_near_gaps", "loiter_hours", "n_encounters", "n_carrier_encounters", "n_port_visits"]
display(nodes[cols].describe().T.round(2))
summary = {"region": REGION_ID, "start": START, "end": END, "n_events": len(ev), "n_vessels": int(ev.vessel_id.nunique()),
           "events_by_type": ev.event_type.value_counts().to_dict(), "sar_unmatched_cell_days": len(sar),
           "graph_nodes": len(nodes), "graph_edges": edges.edge_type.value_counts().to_dict(), "caveat": REAL_CAVEAT}
(METRIC_DIR / "real_data_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("saved to", REAL_DIR)
if IN_COLAB:   # copy to Google Drive, replacing the older copy, so notebooks 04b and 05 use this period
    import shutil
    DRIVE_REAL = DRIVE_DATA / "real_gfw"
    DRIVE_REAL.mkdir(parents=True, exist_ok=True)
    for f in REAL_DIR.iterdir():
        if f.is_file():
            shutil.copy2(f, DRIVE_REAL / f.name)
    print("copied to Google Drive:", DRIVE_REAL, "| period", START, "->", END)

## What this real data can and cannot tell us

**Can:**
- show real fishing activity, vessel meetings, loitering and AIS-off gaps in the Gulf of Mannar and Palk Bay, where they happen relative to protected areas and the maritime boundary, and how they change over time;
- show independent **radar evidence** of unidentified vessels near AIS gaps;
- build a real interaction graph for the model.

**Cannot:**
- prove any vessel acted illegally, since gaps have many innocent causes (coverage, equipment, port stays);
- see vessels that carry no AIS, except through radar;
- provide confirmed labels: none exist publicly for these waters.

**Next:** train the temporal graph model on this real graph. Without confirmed labels, it will be trained to **rank vessel-weeks for investigation** using self-supervised learning plus weak real evidence, and it will be validated against independent evidence (radar detections, GFW's intentional-disabling flag) that is held out from its inputs.